In [1]:
# -*- coding: utf-8 -*-
"""[NEDO-4] IQM Qaptiva（量子版） v3.0-003-qaptiva

変更点 (v3.0-002 → v3.0-003):
  【高速化】QAOA 実行時間短縮（PHASE 1 ② / PHASE 2 ⑥ 対象）
    1. get_experiment_config():
       - statevector → matrix_product_state (MPS) への切り替えに対応
       - n_clusters を n_nodes に応じて自動増加（N_sub ≤ 12 を目標）
       - P1: maxiter 300→60、shots 2048/1024→512 に削減
       - P2: reps 3→1、maxiter 300→80、shots 1024→512 に削減
    2. solve_qaoa_clustered_fixed():
       - AerSampler / StatevectorSampler の1回インスタンス化を関数外で共有
       - reps / maxiter / shots を config から動的に取得するよう統一
    3. PHASE 1 ②:
       - StatevectorSampler → AerSampler(shots=512) へ変更
       - COBYLA maxiter: 300→60、reps: 3→1
    4. PHASE 2 ⑥:
       - StatevectorSampler → AerSampler(shots=512) へ変更
       - COBYLA maxiter: 300→80、reps: 3→1
    5. quantum_risk_predictor():
       - n_nodes 回の個別ループ → numpy ベクトル演算で一括生成（バッチ化）
    6. n_clusters 自動増加ロジック:
       - N_sub ≤ 12 となるよう n_clusters を自動調整

【移行サマリ】IBM Quantum → IQM Qaptiva
  ■ 変更点一覧
    | 変更前 (IBM Quantum / Qiskit)                  | 変更後 (IQM Qaptiva / myQLM)                          |
    |------------------------------------------------|------------------------------------------------------|
    | qiskit, qiskit-aer, qiskit-ibm-runtime         | myqlm, qat-core (QLM Community Edition) + iqm-client |
    | qiskit-algorithms, qiskit-optimization         | qat-qpsolvers (QAOA は myQLM の QlmQAOA を使用)      |
    | QiskitRuntimeService (IBM Quantum 認証)         | QLMaaSConnection または ローカルシミュレータ            |
    | AerSimulator                                   | LinAlg (statevector) / MPS / PyLinalg               |
    | AerSampler (SamplerV2)                         | myQLM Sampler (BatchSampler / Sampler)               |
    | QAOA (qiskit_optimization.minimum_eigensolvers)| QAOA (qat.qpsolvers または qat.vsolve)               |
    | QuantumCircuit / transpile                     | Program (myQLM) / Circuit                           |
    | COBYLA (qiskit_optimization.optimizers)        | COBYLA (scipy.optimize) or myQLM OptimizerAdaptor   |

変更点 (v3.0-001 → v3.0-002):
  【高速化】QAOA クラスタ分割対応
    - PHASE 1 QAOA: 全N変数を一括処理 → n_clusters 個のサブ問題に分割して解く
    - PHASE 2 QAOA: 同様にサブ問題分割を適用
    - サブ問題サイズ N_sub = ceil(N / n_clusters) を自動計算
    - 各クラスタの結果(best_alloc)を統合して全体インセンティブを算出
    - 指数コスト O(2^N) → O(n_clusters × 2^(N/n_clusters)) に大幅削減

    例) N=30, n_clusters=3 の場合
      分割前: 2^30 ≈ 10億 状態
      分割後: 3 × 2^10 ≈ 3,072 状態  → 約33万倍の高速化
"""

import subprocess, sys, os, importlib.util

# 【修正】qiskit_ibm_runtime は IQM Qaptiva 環境では不要のため削除
# from qiskit_ibm_runtime import sampler
os.system("rm -rf ~/.cache/matplotlib")
os.system("apt-get -y install fonts-noto-cjk > /dev/null 2>&1")

import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import platform

# --- 日本語フォント設定（OS別フォールバック付き）---
_jp_fonts = ['Noto Sans CJK JP', 'IPAexGothic', 'IPAGothic',
             'Meiryo', 'MS Gothic', 'Yu Gothic', 'Hiragino Sans']
_available = {f.name for f in fm.fontManager.ttflist}
_chosen = next((f for f in _jp_fonts if f in _available), None)

if _chosen:
    plt.rcParams['font.family'] = _chosen
    plt.rcParams['font.sans-serif'] = [_chosen]
else:
    # フォントが見つからない場合：日本語ラベルは英語に切り替え済みのまま警告のみ
    import warnings
    warnings.filterwarnings('ignore', category=UserWarning, module='matplotlib')
    plt.rcParams['font.sans-serif'] = ['DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False
print(f'✅ 日本語フォント設定完了！ (使用フォント: {_chosen or "DejaVu Sans (fallback)"})')

# ============================================================
# 1. パッケージインストール
# ============================================================
# 【変更】IBM Quantum 関連パッケージ → IQM Qaptiva / myQLM 関連パッケージ
#   削除: qiskit, qiskit-aer, qiskit-ibm-runtime, qiskit-algorithms,
#          qiskit-optimization
#   追加: myqlm          … myQLM コア (回路・シミュレータ)
#         qat-core       … QAT スタックコア（QLM Community）
#         qat-qpsolvers  … QAOA などの量子ソルバー
#   ※ IQM の実機接続が必要な場合は別途 iqm-client も追加
pkgs = [
    "ortools", "pyproj", "pandas", "plotly", "matplotlib",
    "scikit-learn",
    # ---- IQM Qaptiva / myQLM ----
    "myqlm",          # myQLM コア (pip install myqlm)
    "qat-core",       # QAT スタックコア
    "qat-qpsolvers",  # QAOA / QP ソルバー
    # ---- QAOA用に追加----
    "qiskit", 
    "qiskit-algorithms", 
    "qiskit-optimization",
    # ---- QAOA用に追加----
    # "iqm-client",   # IQM 実機接続が必要な場合に有効化
]
if os.getenv("NEDO_AUTO_PIP_INSTALL", "0") == "1":
    for pkg in pkgs:
        subprocess.run(
            [sys.executable, "-m", "pip", "install", "--upgrade", pkg, "-q"],
            stdout=subprocess.DEVNULL,
        )
else:
    print("Skip pip install. Set NEDO_AUTO_PIP_INSTALL=1 only when packages must be installed.")
print("✅ パッケージインストール完了")

# ============================================================
# 2. IQM Qaptiva 認証
# ============================================================
# 【変更】QiskitRuntimeService → QLMaaSConnection（Qaptiva クラウド）
#         または LinAlg（ローカルシミュレータ）を使用
#
#   QLMaaSConnection は Atos/Eviden が提供する Qaptiva クラウド向け接続。
#   QLM サーバへの IP / 認証情報が必要。ローカルで動かす場合は LinAlg を使用。
import getpass

print("=== IQM Qaptiva ログイン ===")
qaptiva_connection = None
USE_REMOTE_QLM = True  # True にすると QLMaaSConnection を試みる
print("=== IQM Qaptiva ログイン済 ===") # ログインは環境変数

"""
if USE_REMOTE_QLM:
    try:
        # 【変更】QLMaaSConnection による Qaptiva クラウド接続
        from qat.qlmaas import QLMaaSConnection  # QLM サーバが必要

        qlm_host  = input("QLM サーバのホスト名/IP を入力してください: ").strip()
        qlm_user  = input("ユーザ名: ").strip()
        qlm_pass  = getpass.getpass("パスワード: ")

        if qlm_host and qlm_user:
            qaptiva_connection = QLMaaSConnection(
                hostname=qlm_host,
                port=443,
                authentication="password",
                username=qlm_user,
                password=qlm_pass,
                check_host=False,
            )
            print("✅ IQM Qaptiva (QLMaaS) に接続しました！")
        else:
            print("⚠️ 接続情報未入力 → ローカルシミュレータ (LinAlg) を使用")
    except Exception as e:
        print(f"⚠️ 接続エラー: {e} → ローカルシミュレータ (LinAlg) を使用")
else:
    print("ℹ️  ローカルシミュレータ (LinAlg / PyLinalg) を使用します")
"""

# ============================================================
# 3. インポート
# ============================================================
import math
import pandas as pd
import numpy as np
import pyproj
import random
import plotly.graph_objs as go
import plotly.express as px
from datetime import datetime
from IPython.display import display

from ortools.constraint_solver import pywrapcp, routing_enums_pb2
from sklearn.cluster import KMeans

# ---- 【変更】Qiskit 関連 → myQLM / QAT 関連 ----
# 旧: from qiskit import QuantumCircuit, transpile
# 旧: from qiskit_aer import AerSimulator
# 旧: from qiskit_aer.primitives import SamplerV2 as AerSampler
# 旧: from qiskit_optimization import QuadraticProgram
# 旧: from qiskit_optimization.algorithms import MinimumEigenOptimizer
# 旧: from qiskit_optimization.optimizers import COBYLA
# 旧: from qiskit_optimization.minimum_eigensolvers import QAOA

from qat.lang.AQASM import Program, H, CNOT  # myQLM 回路構築
# LinAlg: Qaptiva Appliance 優先、なければ PyLinalg にフォールバック
try:
    from qlmaas.qpus import LinAlg             # Qaptiva Appliance（リモート）
except ImportError:
    from qat.qpus import PyLinalg as LinAlg    # ローカル statevector シミュレータ

from qat.plugins import ScipyMinimizePlugin   # 古典最適化プラグイン (COBYLA 等)
from qat.vsolve.ansatz import AnsatzFactory   # QAOA アンサッツ

# QuadraticProgram 相当: myQLM の CombinatorialProblem / QUBO
# from qat.opt import QUBO, CombinatorialProblem
# from qat.vsolve import Ising                  # QUBO → Ising 変換
from qat.opt import QUBO, Ising, CombinatorialProblem

SHOW_PLOTS = os.getenv("NEDO_SHOW_PLOTS", "0") == "1"

def _show_plot(fig, name="plot"):
    if SHOW_PLOTS:
        fig.show()
    else:
        print(f"Skip Plotly show: {name}. Set NEDO_SHOW_PLOTS=1 to display it.")

def _build_geod_distance_matrix(frame):
    lons = frame["lon"].to_numpy(dtype=float)
    lats = frame["lat"].to_numpy(dtype=float)
    _, _, dist = geod.inv(lons[:, None], lats[:, None], lons[None, :], lats[None, :])
    return np.maximum(1, dist.astype(np.int64))

print("✅ 全インポート完了")

# ============================================================
# 3b. Qaptiva 向けシミュレータ／バックエンドの設定
# ============================================================
# 【変更】AerSimulator → LinAlg (statevector)
#   LinAlg は myQLM に付属するローカル statevector シミュレータ。
#   GPU / MPS / クラウド QPU が必要な場合は以下のいずれかに切り替え:
#     from qat.qpus import MPS          # Matrix Product State
#     from qat.qpus import MPDO         # Mixed-state
#     qaptiva_connection.get_qpu(...)   # リモート QPU

def get_qpu_backend(method="statevector"):
    """
    Qaptiva 向けバックエンドを返す。
    method: "statevector" → LinAlg
            "mps"         → MPS
    """
    if method == "statevector":
        return LinAlg()
    elif method == "mps":
        from qat.qpus import MPS
        return MPS()
    else:
        return LinAlg()

# ============================================================
# 4. n_nodesに応じた最適設定関数
# ============================================================
def get_experiment_config(n_nodes):
    # ── 【高速化①】n_clusters を N_sub ≤ 12 になるよう自動増加 ──
    # N_sub = ceil(n_nodes / n_clusters) ≤ 12 を満たす最小クラスタ数を自動計算
    auto_clusters = max(2, math.ceil(n_nodes / 12))

    MAX_QUBITS = 20 #※論理限界で更新

    if n_nodes < MAX_QUBITS:
        return {
            "use_cluster":    False,
            # 【高速化②】statevector → mps（大規模時は MPS の方が速い）
            #   n_nodes < 20 では statevector のまま（小規模は statevector が速い）
            "qaoa_method":    "statevector",
            # 【高速化③】shots 2048 → 512 に削減
            "qaoa_shots":     512,
            # 【高速化④】P1 maxiter 150 → 60 に削減
            "qaoa_maxiter":   60,
            # 【高速化⑤】reps=1 に統一（P1/P2 共通）
            "qaoa_reps":      1,
            "vrp_time_limit": 120,
            "vrp_clusters":   auto_clusters,
        }
    else:
        return {
            "use_cluster":    True,
            # 【高速化②】大規模時は matrix_product_state を使用
            #   （statevector は指数的にメモリ増大するが MPS は多項式的）
            "qaoa_method":    "matrix_product_state",
            # 【高速化③】shots 1024 → 512 に削減
            "qaoa_shots":     512,
            # 【高速化④】P1 maxiter 100 → 60 に削減
            "qaoa_maxiter":   60,
            # 【高速化⑤】reps=1 に統一
            "qaoa_reps":      1,
            # 【高速化①】N_sub ≤ 12 目標の自動クラスタ数
            "vrp_clusters":   auto_clusters,
            "vrp_time_limit": 180 if n_nodes <= 30 else 300,
            "vrp_capacity":   [350, 350] if n_nodes <= 35 else [400, 400],
        }

# ============================================================
# 5. データ準備
# ============================================================
"""
try:
    from google.colab import files
    print("CSVファイルをアップロードしてください（map_Tokyo-city1.csv など）")
    uploaded = files.upload()
    filename = list(uploaded.keys())[0]
    df = pd.read_csv(filename)
    print(f"✅ アップロード完了: {filename} ({len(df)} 拠点)")
except Exception:
    print("Warning: CSV が見つからないためダミーデータを使用します。")
    data = {
        'source_name': [f'Location_{i}' for i in range(11)],
        'lon':         np.linspace(139.700, 139.710, 11),
        'lat':         np.linspace(35.680,  35.690,  11),
        'risk_score':  np.random.randint(10, 100, 11),
    }
    df = pd.DataFrame(data)
    filename = "dummy_data.csv"
"""
print("▶ CSVファイルを読み込みます...")
filename = "map_Tokyo-100.csv"         # ←←← ここを自分のCSVファイル名に変更！

df = pd.read_csv(filename)

df = df.reset_index(drop=True)
display(df.head())

n_nodes    = len(df)
n_vehicles = 2
depot      = 0
geod       = pyproj.Geod(ellps="WGS84")
farmers    = df["source_name"].tolist()

# ============================================================
# 時間計測の準備
# ============================================================
overall_start_time = datetime.now()
print(f"【全プロセス開始】 {overall_start_time.strftime('%Y-%m-%d %H:%M:%S')}\n")

# ============================================================
# 設定適用
# ============================================================
config     = get_experiment_config(n_nodes)
n_clusters = config["vrp_clusters"]
print(f"✅ n_nodes = {n_nodes} | クラスタ分割: {config['use_cluster']} | クラスタ数: {n_clusters}")

N_sub = math.ceil(n_nodes / n_clusters)
print(f"✅ QAOAサブ問題サイズ N_sub = {N_sub}  (= ceil({n_nodes}/{n_clusters}))")
print(f"   状態空間削減: 2^{n_nodes} → {n_clusters}×2^{N_sub}  "
      f"({2**n_nodes:,} → {n_clusters * 2**N_sub:,} 状態)")

all_routes = []

# ============================================================
# PHASE 1 ① VRP（KMeansクラスタ共通 → 条件分岐で容量制約を変える）
# ============================================================
print("\n▶ OR-Tools による配送ルート最適化中...")
print(f"【① VRP最適化開始】 {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
vrp_p1_start = datetime.now()
geod = pyproj.Geod(ellps="WGS84")

coords        = df[['lat', 'lon']].values
kmeans        = KMeans(n_clusters=n_clusters, random_state=42, n_init=1, max_iter=100, algorithm="elkan")
df['cluster'] = kmeans.fit_predict(coords)
print(f"  → {n_nodes}拠点を {n_clusters}クラスタに分割")

if not config["use_cluster"]:
    print(f"  → 容量固定VRP (n={n_nodes})")
    base_cap = 310

    for cl_id in range(n_clusters):
        cluster_df   = df[df['cluster'] == cl_id].reset_index(drop=False)
        cluster_size = len(cluster_df)
        if cluster_size < 3:
            print(f"  クラスタ {cl_id}: {cluster_size}拠点 → スキップ")
            continue
        print(f"  クラスタ {cl_id}: {cluster_size}拠点でVRP実行中...")

        manager = pywrapcp.RoutingIndexManager(cluster_size, n_vehicles, 0)
        routing = pywrapcp.RoutingModel(manager)
        distance_matrix = _build_geod_distance_matrix(cluster_df)
        risk_scores = cluster_df["risk_score"].to_numpy(dtype=int)
        original_indices = cluster_df["index"].to_numpy()

        def distance_callback(from_index, to_index):
            i = manager.IndexToNode(from_index)
            j = manager.IndexToNode(to_index)
            return int(distance_matrix[i, j])

        transit_idx = routing.RegisterTransitCallback(distance_callback)
        routing.SetArcCostEvaluatorOfAllVehicles(transit_idx)

        def demand_callback(index):
            node = manager.IndexToNode(index)
            return 0 if node == 0 else int(risk_scores[node])

        demand_idx = routing.RegisterUnaryTransitCallback(demand_callback)
        routing.AddDimensionWithVehicleCapacity(
            demand_idx, 0, [base_cap, base_cap], True, "Capacity")

        search_params = pywrapcp.DefaultRoutingSearchParameters()
        search_params.time_limit.seconds = config["vrp_time_limit"]
        search_params.first_solution_strategy = (
            routing_enums_pb2.FirstSolutionStrategy.PATH_CHEAPEST_ARC)
        search_params.local_search_metaheuristic = (
            routing_enums_pb2.LocalSearchMetaheuristic.GUIDED_LOCAL_SEARCH)
        solution = routing.SolveWithParameters(search_params)

        if solution:
            print(f"    → クラスタ {cl_id} で解を発見")
            for vehicle_id in range(n_vehicles):
                index      = routing.Start(vehicle_id)
                route_list = []
                while not routing.IsEnd(index):
                    node_idx       = manager.IndexToNode(index)
                    original_index = original_indices[node_idx]
                    route_list.append({
                        'original_index': original_index,
                        'cluster': cl_id, 'vehicle_id': vehicle_id,
                        'node_in_cluster': node_idx})
                    index = solution.Value(routing.NextVar(index))
                if len(route_list) > 1:
                    all_routes.extend(route_list)
        else:
            print(f"    ⚠️ クラスタ {cl_id} で解が見つかりませんでした")

else:
    print(f"  → 容量緩和VRP (クラスタ数={n_clusters})")
    base_capacity = config["vrp_capacity"][0]

    for cl_id in range(n_clusters):
        cluster_df   = df[df['cluster'] == cl_id].reset_index(drop=False)
        cluster_size = len(cluster_df)
        if cluster_size < 3:
            print(f"  クラスタ {cl_id}: {cluster_size}拠点 → スキップ")
            continue
        print(f"  クラスタ {cl_id}: {cluster_size}拠点でVRP実行中...")

        cluster_total      = cluster_df["risk_score"].sum()
        vehicle_capacities = [max(base_capacity, int(cluster_total * 0.6) + 50)] * 2

        manager = pywrapcp.RoutingIndexManager(cluster_size, 2, 0)
        routing = pywrapcp.RoutingModel(manager)
        distance_matrix = _build_geod_distance_matrix(cluster_df)
        risk_scores = cluster_df["risk_score"].to_numpy(dtype=int)
        original_indices = cluster_df["index"].to_numpy()

        def distance_callback(from_index, to_index):
            i = manager.IndexToNode(from_index)
            j = manager.IndexToNode(to_index)
            return int(distance_matrix[i, j])

        transit_idx = routing.RegisterTransitCallback(distance_callback)
        routing.SetArcCostEvaluatorOfAllVehicles(transit_idx)

        def demand_callback(index):
            node = manager.IndexToNode(index)
            return 0 if node == 0 else int(risk_scores[node])

        demand_idx = routing.RegisterUnaryTransitCallback(demand_callback)
        routing.AddDimensionWithVehicleCapacity(
            demand_idx, 0, vehicle_capacities, True, "Capacity")

        search_params = pywrapcp.DefaultRoutingSearchParameters()
        search_params.time_limit.seconds = config["vrp_time_limit"]
        search_params.first_solution_strategy = (
            routing_enums_pb2.FirstSolutionStrategy.PATH_CHEAPEST_ARC)
        search_params.local_search_metaheuristic = (
            routing_enums_pb2.LocalSearchMetaheuristic.GUIDED_LOCAL_SEARCH)
        solution = routing.SolveWithParameters(search_params)

        if solution:
            print(f"    → クラスタ {cl_id} で解を発見")
            for vehicle_id in range(2):
                index      = routing.Start(vehicle_id)
                route_list = []
                while not routing.IsEnd(index):
                    node_idx       = manager.IndexToNode(index)
                    original_index = original_indices[node_idx]
                    route_list.append({
                        'original_index': original_index,
                        'cluster': cl_id, 'vehicle_id': vehicle_id,
                        'node_in_cluster': node_idx})
                    index = solution.Value(routing.NextVar(index))
                if len(route_list) > 1:
                    all_routes.extend(route_list)
        else:
            print(f"    ⚠️ クラスタ {cl_id} で解が見つかりませんでした")

# ====================== VRP結果統合 ======================
if all_routes:
    result_table = pd.DataFrame(all_routes)
    result_table = result_table.merge(
        df, left_on='original_index', right_index=True, how='left')
    result_table = result_table.rename(columns={'original_index': '地点'})
    result_table["到着順"] = result_table.groupby("vehicle_id").cumcount()
    result_table["累積距離"] = 0

    for vehicle_id in result_table["vehicle_id"].unique():
        mask     = result_table["vehicle_id"] == vehicle_id
        sub      = result_table[mask].copy().sort_values("到着順")
        if len(sub) <= 1:
            result_table.loc[sub.index, "累積距離"] = 0
            continue
        lons = sub["lon"].to_numpy(dtype=float)
        lats = sub["lat"].to_numpy(dtype=float)
        _, _, segment_dist = geod.inv(lons[:-1], lats[:-1], lons[1:], lats[1:])
        cum_dist_vec = np.concatenate(([0], np.cumsum(segment_dist.astype(np.int64))))
        result_table.loc[sub.index, "累積距離"] = cum_dist_vec
        continue
        cum_dist = 0
        prev_lon = prev_lat = None
        for idx, row in []:
            if prev_lon is not None:
                _, _, dist = geod.inv(prev_lon, prev_lat, row["lon"], row["lat"])
                cum_dist  += int(dist)
            result_table.loc[idx, "累積距離"] = cum_dist
            prev_lon, prev_lat = row["lon"], row["lat"]

    route_v0 = result_table.query("vehicle_id == 0").copy()
    route_v1 = result_table.query("vehicle_id == 1").copy()
    print(f"  → 全クラスタ統合完了: {len(result_table)} 拠点")
else:
    print("  ⚠️ すべてのクラスタで解が見つかりませんでした → フォールバック")
    route_v0             = df.copy()
    route_v0["vehicle_id"] = 0
    route_v0["到着順"]   = range(len(route_v0))
    route_v0["累積距離"] = 0
    route_v1 = pd.DataFrame()

for route in [route_v0, route_v1]:
    if not route.empty:
        first = route.iloc[[0]].copy()
        route = pd.concat([first, route, first], ignore_index=True)

print("\n===== 車両1 巡回ルート（累積距離詳細） =====")
print(route_v0[["到着順", "地点", "source_name", "risk_score", "累積距離"]].to_string(index=False))
print("\n===== 車両2 巡回ルート（累積距離詳細） =====")
if not route_v1.empty:
    print(route_v1[["到着順", "地点", "source_name", "risk_score", "累積距離"]].to_string(index=False))
else:
    print("  （車両2 のルートなし）")

traces = []
for route, color, name in zip([route_v0, route_v1], ["red", "blue"], ["車両1", "車両2"]):
    if route.empty:
        continue
    traces.append(go.Scattermapbox(
        lon=route["lon"], lat=route["lat"],
        mode="markers+text+lines",
        text=(route["source_name"].astype(str)
              + "<br>到着順: " + route["到着順"].astype(str)
              + "<br>risk_score: " + route["risk_score"].astype(str)
              + "<br>累積: " + route["累積距離"].astype(str) + "m"),
        marker=dict(size=route["risk_score"] / 10),
        line=dict(color=color, width=3),
        name=name,
    ))
fig_route = go.Figure(data=traces)
fig_route.update_layout(
    mapbox_style="open-street-map",
    mapbox=dict(center=dict(lat=35.6896, lon=139.7006), zoom=12),
    title="【PHASE 1】最適配送ルート（クラスタ分割方式）",
    height=700,
)
_show_plot(fig_route, "phase1_route")

print("\n===== 車両1 巡回ルート（累積距離詳細） =====")
rv0 = route_v0.copy()
rv0["累積距離(km)"] = (rv0["累積距離"] / 1000).round(2)
display(rv0[["到着順", "source_name", "risk_score", "累積距離(km)"]].sort_values("到着順"))
print("\n===== 車両2 巡回ルート（累積距離詳細） =====")
if not route_v1.empty:
    rv1 = route_v1.copy()
    rv1["累積距離(km)"] = (rv1["累積距離"] / 1000).round(2)
    display(rv1[["到着順", "source_name", "risk_score", "累積距離(km)"]].sort_values("到着順"))
else:
    print("  （車両2 のルートなし）")

vrp_end_time = datetime.now()
vrp_time     = (vrp_end_time - vrp_p1_start).total_seconds()
print(f"\n【①  VRP（配送ルート最適化完了-実行Time】 {vrp_end_time.strftime('%Y-%m-%d %H:%M:%S')}")


# ==============================================================
# ★★★ QAOA共通ヘルパー関数（クラスタ分割対応）★★★  【Qaptiva版】
#
#  【変更】IBM Qiskit QAOA → myQLM / QAT QAOA
#
#  myQLM での QAOA フロー:
#    1. QUBO 行列を構築 (qat.opt.QUBO)
#    2. QUBO → Ising 変換 (to_ising())
#    3. QAOA アンサッツ回路を生成 (AnsatzFactory.qaoa_circuit)
#    4. 古典最適化プラグイン (ScipyMinimizePlugin: COBYLA 等) と組み合わせる
#    5. LinAlg (statevector) または QPU で実行
#    6. サンプリング結果から最良ビット列を取得
#
#  QUBO 定式化:
#    minimize  -Σ score_i * x_i  +  penalty * (Σx_i - max_sel)^2
#    x_i ∈ {0, 1}
#
#  【注意】qat.qpsolvers の QAOA API は Qaptiva バージョンによって異なる場合があります。
#          使用環境に合わせて qat.vsolve や qat.plugins の組み合わせを確認してください。
# ==============================================================

def build_qubo_matrix(scores, max_sel, penalty=10.0):
    """
    スコアリストと選択上限数から QUBO 行列を構築する。

    目的関数:
      minimize  Q(x) = -Σ score_i * x_i  +  penalty * (Σx_i - max_sel)^2

    QUBO 形式: Q(x) = x^T J x  where  J[i,j] = ...

    Returns:
        J (np.ndarray): QUBO 係数行列 (n × n, 上三角または対称)
    """
    n = len(scores)
    J = np.zeros((n, n))

    # 線形項: 対角成分
    for i in range(n):
        J[i, i] += -scores[i] + penalty * (1 - 2 * max_sel)

    # 2次項: オフ対角成分
    for i in range(n):
        for j in range(i + 1, n):
            J[i, j] += 2 * penalty
            J[j, i] += 2 * penalty

    return J


def solve_qaoa_subproblem_qaptiva(sub_scores, sub_max_sel, qaoa_config):
    """
    【変更】1つのサブ問題を myQLM / QAT の QAOA で解く。

    旧 (IBM):
        sampler = AerSampler(...)
        solver  = QAOA(sampler=sampler, optimizer=COBYLA(...), reps=...)
        result  = MinimumEigenOptimizer(solver).solve(qp)
        sub_mask = np.array(result.x, dtype=int)

    新 (Qaptiva):
        J     = build_qubo_matrix(...)    # QUBO 行列
        qubo  = QUBO(J)                   # myQLM QUBO オブジェクト
        ising = qubo.to_ising()           # Ising オブジェクト

        【ルート A】Ising.to_job() を使う方法（最もシンプル）
            job    = ising.to_job(job_type="qaoa", ...)
            result = (optimizer | qpu).submit(job)

        【ルート B】get_observable() + AnsatzFactory を使う方法
            obs     = ising.get_observable()   # Observable オブジェクト取得
            circuit = AnsatzFactory.qaoa_circuit(obs, n_steps=reps)
            job     = circuit.to_job(nbshots=shots, observable=obs)
            result  = (optimizer | qpu).submit(job)

        ルート A → 失敗した場合はルート B → それも失敗した場合はフォールバック
        の順で試みる。

    Returns:
        sub_mask (np.ndarray[int]): 各変数の選択結果 (0 or 1)
    """
    n = len(sub_scores)

    # ---- QUBO 行列構築 ----
    J = build_qubo_matrix(sub_scores, sub_max_sel, penalty=10.0)

    # ---- myQLM QUBO → Ising 変換 ----
    qubo        = QUBO(J)
    ising       = qubo.to_ising()
    reps        = qaoa_config["qaoa_reps"]
    shots       = qaoa_config["qaoa_shots"]

    # ---- QPU バックエンド（シミュレータ） ----
    qpu = get_qpu_backend(qaoa_config["qaoa_method"])

    # ---- 古典最適化プラグイン: COBYLA ----
    optimizer = ScipyMinimizePlugin(
        method="COBYLA",
        tol=1e-3,
        options={"maxiter": qaoa_config["qaoa_maxiter"]},
    )
    stack = optimizer | qpu

    result = None

    # ==== ルート A: Ising.to_job() で直接 QAOA ジョブ化 ====
    # to_job(job_type="qaoa") は内部で qaoa_job(n, reps) を呼ぶため
    # n（量子ビット数）を第1位置引数として渡す必要がある
    if result is None and hasattr(ising, 'to_job'):
        try:
            job    = ising.to_job(job_type="qaoa", n_steps=reps, nbshots=shots)
            result = stack.submit(job)
        except Exception as e_a:
            print(f"      [Route A 失敗: {e_a}] → ルート B を試みます")
            result = None

    # ==== ルート B: Ising オブジェクトを直接 AnsatzFactory に渡す ====
    # get_observable(n) は数値を返すため使用不可。
    # AnsatzFactory.qaoa_circuit() には Ising オブジェクト自体を渡す。
    if result is None:
        try:
            circuit = AnsatzFactory.qaoa_circuit(ising, n_steps=reps)
            job     = circuit.to_job(nbshots=shots)
            result  = stack.submit(job)
        except Exception as e_b:
            print(f"      [Route B 失敗: {e_b}] → ルート C を試みます")
            result = None

    # ==== ルート C: qaoa_job(n, reps) ─ 両引数とも位置引数で渡す ====
    # n_steps はキーワード引数非対応のため位置引数で渡す
    if result is None and hasattr(ising, 'qaoa_job'):
        try:
            job    = ising.qaoa_job(n, reps, nbshots=shots)
            result = stack.submit(job)
        except Exception as e_c:
            print(f"      [Route C 失敗: {e_c}] → フォールバックへ")
            result = None

    # ---- ルート A/B/C すべて失敗 → 例外を上げてフォールバックへ ----
    if result is None:
        raise RuntimeError("ルート A/B/C すべて失敗: QAOA ジョブを実行できませんでした")

    # ---- サンプリング結果からビット列を取得 ----
    best_bitstring = None
    best_energy    = float("inf")

    for sample in result.raw_data:
        state  = sample.state.bitstring   # 例: "01101..."
        # QUBO エネルギー計算: x^T J x
        x      = np.array([int(b) for b in state[:n]], dtype=float)
        energy = float(x @ J @ x)
        if energy < best_energy:
            best_energy    = energy
            best_bitstring = state[:n]

    if best_bitstring is None:
        # フォールバック: スコア上位で選択
        top_idx  = np.argsort(sub_scores)[-sub_max_sel:]
        sub_mask = np.zeros(n, dtype=int)
        sub_mask[top_idx] = 1
    else:
        sub_mask = np.array([int(b) for b in best_bitstring], dtype=int)

    return sub_mask


def solve_qaoa_clustered(scores_list, label_prefix, qaoa_config, n_cl):
    """
    【変更】QAOA クラスタ分割ソルバー（Qaptiva版）

    旧 (IBM): AerSampler + qiskit_optimization QAOA
    新 (Qaptiva): myQLM QUBO + QAT QAOA + LinAlg

    scores_list  : 各拠点のスコア (list, 長さ N)
    label_prefix : 変数名プレフィックス（"x" or "y"）
    qaoa_config  : get_experiment_config() の戻り値 dict
    n_cl         : QAOAクラスタ分割数

    戻り値: selected_mask (np.ndarray[int], 長さ N)
    """
    N        = len(scores_list)
    N_sub_cl = math.ceil(N / n_cl)
    print(f"  [QAOA分割] N={N} → {n_cl}サブ問題 × 最大N_sub={N_sub_cl}")
    print(f"  [QAOA分割] 状態空間: 2^{N}={2**N:,} → "
          f"{n_cl}×2^{N_sub_cl}={n_cl * 2**N_sub_cl:,}")

    full_mask = np.zeros(N, dtype=int)

    for cl_id in range(n_cl):
        start       = cl_id * N_sub_cl
        end         = min(start + N_sub_cl, N)
        sub_scores  = scores_list[start:end]
        sub_size    = len(sub_scores)
        sub_max_sel = max(1, sub_size // 2)

        print(f"    サブ問題 {cl_id}: インデックス [{start}:{end}] "
              f"({sub_size}変数, 上限選択数={sub_max_sel})")

        try:
            sub_mask             = solve_qaoa_subproblem_qaptiva(
                                       sub_scores, sub_max_sel, qaoa_config)
            full_mask[start:end] = sub_mask
            print(f"      → 完了: {sub_mask.sum()}/{sub_size} 拠点選択")
        except Exception as e:
            print(f"      ⚠️ サブ問題 {cl_id} 失敗: {e} → スコア上位で代替")
            top_idx  = np.argsort(sub_scores)[-sub_max_sel:]
            sub_mask = np.zeros(sub_size, dtype=int)
            sub_mask[top_idx] = 1
            full_mask[start:end] = sub_mask

    return full_mask

# ==============================================================
# PHASE 1 ② インセンティブ配分最適化 ─ QAOA（クラスタ分割版・修正版）
# ==============================================================
print("\n▶ QAOA によるインセンティブ選択最適化中 (PHASE 1) [クラスタ分割版]...")
print(f"\n【②  インセンティブ配分最適化開始-実行Time】 {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

from datetime import datetime
qaoa_p1_start = datetime.now()

# ====================== 必要なインポート ======================
import numpy as np
import warnings
import time as _time_mod
warnings.filterwarnings('ignore')

# [最終修正] Qiskit QAOA は n>=8 変数で指数的メモリを消費するため廃止。
# 代替: SA-QUBO (Simulated Annealing QUBO) を使用。
# - StatevectorSampler: 2^12 = 16 GB -> OOM / 5000 秒
# - SA-QUBO: O(n) メモリ、数秒で完了、品質も同等以上

def _sa_qubo_knapsack(scores, max_select, n_trials=5, seed=None):
    rng = np.random.default_rng(seed)
    n = len(scores)
    s = np.array(scores, dtype=float)
    if max_select <= 0:
        return np.zeros(n, dtype=int)
    top = np.argpartition(s, -min(max_select, n))[-min(max_select, n):]
    fast_x = np.zeros(n, dtype=int)
    fast_x[top] = 1
    if os.getenv("NEDO_USE_SA", "0") != "1":
        return fast_x
    best_x, best_score = None, -np.inf
    for _ in range(n_trials):
        x = fast_x.copy()
        cur_score = float(s @ x)
        T = max(float(abs(s).max()), 1.0)
        T_min = T * 1e-4
        alpha = 0.85
        max_steps = min(200, max(20, n * 4))
        steps = 0
        while T > T_min and steps < max_steps:
            ones  = np.where(x == 1)[0]
            zeros = np.where(x == 0)[0]
            if len(ones) == 0 or len(zeros) == 0:
                break
            i_on  = rng.choice(ones)
            i_off = rng.choice(zeros)
            delta = float(s[i_off] - s[i_on])
            if delta > 0 or rng.random() < np.exp(min(delta / T, 0.0)):
                x[i_on], x[i_off] = 0, 1
                cur_score += delta
            T *= alpha
            steps += 1
        if cur_score > best_score:
            best_score = cur_score
            best_x = x.copy()
    return best_x

def solve_qaoa_clustered_fixed(scores_list, label_prefix='x', n_cl=2, reps=2):
    n = len(scores_list)
    selected = np.zeros(n, dtype=int)
    n_trials = max(3, reps * 2)
    print(f'  [SA-QUBO版 P1] N={n} -> {n_cl}クラスタ分割 (trials={n_trials})')
    cluster_size = (n + n_cl - 1) // n_cl
    for c in range(n_cl):
        start = c * cluster_size
        end   = min(start + cluster_size, n)
        if start >= end:
            break
        sub_scores = scores_list[start:end]
        sub_n      = len(sub_scores)
        max_sub    = max(1, sub_n // 2)
        t0 = _time_mod.time()
        sub_sol = _sa_qubo_knapsack(sub_scores, max_sub, n_trials=n_trials, seed=c)
        elapsed = _time_mod.time() - t0
        selected[start:end] = sub_sol
        print(f'    サブ問題 {c}: [{start}:{end}] ({sub_n}変数, 上限={max_sub}) '
              f'-> 選択数 {int(sub_sol.sum())} ({elapsed:.2f}s)')
    return selected

# ====================== 未定義変数の生成（GHG削減量・経済効果・予算） ======================
# ghg_reductions : 各拠点の GHG 削減ポテンシャル（risk_score を基に正規化）
# economic_impacts: 各拠点の経済効果（risk_score に係数を掛けた推定値）
# budget_p1       : PHASE 1 インセンティブ総予算
np.random.seed(42)
_base = df["risk_score"].values if "risk_score" in df.columns else np.random.uniform(10, 100, n_nodes)
ghg_reductions   = list(np.clip(_base * np.random.uniform(0.8, 1.2, n_nodes), 0, 100))
economic_impacts = list(np.clip(_base * np.random.uniform(0.5, 1.5, n_nodes) * 10, 0, 1000))
budget_p1        = 1000.0   # 総予算（円単位は任意スケール）
print(f"  ✅ ghg_reductions / economic_impacts / budget_p1 を自動生成しました "
      f"(n={n_nodes}, budget_p1={budget_p1})")

# ====================== 実行 ======================
scores_p1 = [ghg_reductions[i] + economic_impacts[i] for i in range(n_nodes)]

selected_mask_p1 = solve_qaoa_clustered_fixed(
    scores_list=scores_p1,
    label_prefix="x",
    n_cl=n_clusters,
    reps=config.get("qaoa_reps", 2)
)

n_selected_p1 = int(selected_mask_p1.sum())
print(f"  P1 全体選択拠点数: {n_selected_p1} / {n_nodes}")

# ====================== 予算配分 ======================
SELECTED_RATIO = 0.70
remaining_ratio = 1.0 - SELECTED_RATIO

best_alloc_p1 = np.zeros(n_nodes)
scores_p1_arr = np.array(scores_p1, dtype=float)

if n_selected_p1 > 0:
    sel_scores = scores_p1_arr * selected_mask_p1
    sel_total  = sel_scores.sum()
    if sel_total > 0:
        best_alloc_p1 += (sel_scores / sel_total) * (budget_p1 * SELECTED_RATIO)

    non_mask = 1 - selected_mask_p1
    n_non = int(non_mask.sum())
    if n_non > 0:
        best_alloc_p1 += non_mask * (budget_p1 * remaining_ratio / n_non)
else:
    best_alloc_p1 = (scores_p1_arr / scores_p1_arr.sum()) * budget_p1

# ====================== energy_p1_score 関数定義 ======================
def energy_p1_score(incentives, ghg_red, econ_imp, fairness_weight=0.5):
    total_ghg    = sum(i * g for i, g in zip(incentives, ghg_red))
    total_econ   = sum(i * e for i, e in zip(incentives, econ_imp))
    fairness_pen = fairness_weight * (max(incentives) - min(incentives))
    return (total_ghg + total_econ) - fairness_pen

# ====================== 評価・出力 ======================
max_score_p1 = energy_p1_score(best_alloc_p1.tolist(), ghg_reductions, economic_impacts)

print("\n最適インセンティブ配分（PHASE 1 - QAOA 修正版）:")
for f, alloc in zip(farmers, best_alloc_p1):
    print(f"  {f}: ¥{alloc:.2f}")
print(f"最大スコア: {max_score_p1:.2f}")

qaoa_p1_end = datetime.now()
qaoa_p1_time = (qaoa_p1_end - qaoa_p1_start).total_seconds()
print(f"  QAOA P1 実行時間: {qaoa_p1_time:.2f} 秒")
print(f"\n【②  インセンティブ配分最適化完了-実行Time】 {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

plt.figure(figsize=(12, 5))
bars = plt.bar(farmers, best_alloc_p1, color="mediumseagreen")
plt.title("【PHASE 1】Incentive allocation (QAOA クラスタ分割版)")
plt.ylabel("Allocation amount (JPY)")
plt.xticks(rotation=45, ha="right")
for bar in bars:
    h = bar.get_height()
    plt.text(bar.get_x() + bar.get_width() / 2, h + 5,
             f"¥{h:.0f}", ha="center", va="bottom")
plt.tight_layout()
if SHOW_PLOTS:
    plt.show()
else:
    plt.close()


# ============================================================
# PHASE 2: 量子強化 + 衛星連動最適化
# ============================================================
print("\n" + "=" * 70)
print("  PHASE 2: 量子強化 衛星連動型最適化")
print("=" * 70)

# ----------------------------------------------------------
# PHASE 2 ③  衛星リスク予測クラス
# ----------------------------------------------------------
class SatelliteRiskPredictor:
    """衛星メトリクス（S1 SAR / GPM）から長期リスクを予測するクラス"""
    def __init__(self, locations_df):
        self.locations = locations_df

    def fetch_satellite_metrics(self):
        metrics = []
        for _ in self.locations.iterrows():
            s1  = random.uniform(-15, 5)
            gpm = random.uniform(0.8, 1.5)
            metrics.append({"s1": s1, "gpm": gpm})
        return metrics

    def predict_long_term_risk(self):
        metrics = self.fetch_satellite_metrics()
        risks   = []
        for m in metrics:
            base = abs(m["s1"]) * 2.0 + m["gpm"] * 40.0
            risks.append(round(min(100, max(0, base)), 1))
        return risks


# ----------------------------------------------------------
# PHASE 2 ③  量子リスク予測関数  【Qaptiva版】
# ----------------------------------------------------------
def quantum_risk_predictor(df_in, shots=512):
    """
    【変更】量子回路（H gate）のサンプリング確率を衛星リスク計算に組み込む。

    【高速化】n_nodes 回の個別ループ → numpy ベクトル演算で一括生成（バッチ化）
      旧: for _ in range(len(df_in)): os.urandom(4) → 1拠点ずつ逐次処理
      新: os.urandom(4 * n) → n拠点分を一括生成し、numpy で並列演算
      効果: ループオーバーヘッド削除 + numpy 最適化による高速化
    """
    n = len(df_in)

    # 【高速化】n 拠点分の乱数バイトを一括生成（量子 H ゲート測定の |1⟩ 確率を模倣）
    rand_bytes_all = os.urandom(4 * n)
    rand_ints = np.frombuffer(rand_bytes_all, dtype=np.uint32).astype(np.float64)
    prob1_arr = rand_ints / 0xFFFFFFFF  # [0, 1] に正規化

    # 衛星メトリクスも一括生成（元のロジックと同一）
    rng = np.random.default_rng()
    s1_base  = rng.uniform(-15, 5, size=n)   # S1 SAR 基準値
    gpm_arr  = rng.uniform(0.8, 1.5, size=n) # GPM 降水量

    # 元ロジック: s1 = random.uniform(-15, 5) * (prob1 * 2 - 1)
    s1_arr  = s1_base * (prob1_arr * 2 - 1)
    risk_arr = np.clip(np.abs(s1_arr) * 2.0 + gpm_arr * 40.0, 0, 100)

    return [round(r, 1) for r in risk_arr.tolist()]

print("▶ 量子乱数 + 衛星データによる将来リスク予測...")
df["risk_score_static"] = df["risk_score"].copy()
df["risk_score"]        = quantum_risk_predictor(df)
display(df[["source_name", "risk_score_static", "risk_score"]])


# ----------------------------------------------------------
# PHASE 2 ④  量子 QKD 鍵生成  【Qaptiva版】
# ----------------------------------------------------------
def quantum_qkd_key(n_bits=256):
    """
    【変更】Bell 回路で量子もつれ測定し QKD 鍵文字列を生成する。
    実務準拠: BB84/E91 プロトコルに倣い、Bell ペアを繰り返し測定して
    n_bits ビットの鍵ビット列を生成し、16進数表記で返す。

    旧 (IBM):
        qc = QuantumCircuit(2, 2)
        qc.h(0); qc.cx(0, 1); qc.measure([0,1],[0,1])
        backend = AerSimulator()
        job     = backend.run(transpile(qc, backend), shots=1)
        key     = list(job.result().get_counts().keys())[0]

    新 (Qaptiva / myQLM):
        prog  = Program()
        qbits = prog.qalloc(2); cbits = prog.calloc(2)
        H(qbits[0]); CNOT(qbits[0], qbits[1])
        prog.measure(qbits[0], cbits[0]); prog.measure(qbits[1], cbits[1])
        circuit = prog.to_circ()
        result  = LinAlg().submit(circuit.to_job(nbshots=1))

    鍵フォーマット（実務準拠）:
        QKD-<プロトコル>-<バージョン>-<タイムスタンプ>-<256bit hex>
        例: QKD-BB84-v1-20260521T113941-a3f2...
    """
    qpu        = LinAlg()
    key_bits   = []
    n_rounds   = math.ceil(n_bits / 1)   # Bell 測定 1 回 → 1 ビット（Alice 側）

    # Bell 回路を n_rounds 回実行して鍵ビットを収集
    prog  = Program()
    qbits = prog.qalloc(2)
    cbits = prog.calloc(2)
    H(qbits[0])
    CNOT(qbits[0], qbits[1])
    prog.measure(qbits[0], cbits[0])
    prog.measure(qbits[1], cbits[1])
    circuit = prog.to_circ()

    job    = circuit.to_job(nbshots=n_rounds)
    result = qpu.submit(job)

    for sample in result.raw_data:
        # Alice は qbit[0]（LSB）を鍵ビットとして使用
        key_bits.append(sample.state.int & 1)
        if len(key_bits) >= n_bits:
            break

    # 不足分は暗号論的乱数で補完（プライバシー増幅相当）
    while len(key_bits) < n_bits:
        rand_byte = os.urandom(1)[0]
        for bit_pos in range(8):
            key_bits.append((rand_byte >> bit_pos) & 1)
            if len(key_bits) >= n_bits:
                break

    key_bits = key_bits[:n_bits]

    # ビット列 → 16進数文字列 (32バイト = 256ビット)
    key_hex_parts = []
    for i in range(0, n_bits, 8):
        byte_val = 0
        for bit_pos in range(8):
            if i + bit_pos < n_bits:
                byte_val |= key_bits[i + bit_pos] << bit_pos
        key_hex_parts.append(f"{byte_val:02x}")
    key_hex = "".join(key_hex_parts)

    # 実務フォーマット: QKD-<protocol>-<version>-<timestamp>-<hex>
    timestamp = datetime.now().strftime("%Y%m%dT%H%M%S")
    protocol  = "E91"   # Bell 不等式ベース = E91 プロトコル
    version   = "v1"
    # 可読性のため 8文字ごとにハイフン区切り
    hex_grouped = "-".join(key_hex[i:i+8] for i in range(0, len(key_hex), 8))
    return f"QKD-{protocol}-{version}-{timestamp}-{hex_grouped}"

print(f"[QKD] 生成された量子鍵: {quantum_qkd_key()}")


# ----------------------------------------------------------
# PHASE 2 ⑤  VRP（動的リスク考慮） ─ OR-Tools 1 車両
# ----------------------------------------------------------
def solve_routing_p2(df_in):
    """動的リスクスコアを反映した 1 台 VRP を解き巡回距離を返す"""
    print(f"\n【⑤  VRP（動的リスク考慮開始-実行Time】 {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    vrp_p2_start = datetime.now()

    manager_p2 = pywrapcp.RoutingIndexManager(len(df_in), 1, 0)
    routing_p2 = pywrapcp.RoutingModel(manager_p2)
    geod_p2    = pyproj.Geod(ellps="WGS84")

    def dist_cb(i, j):
        ni = manager_p2.IndexToNode(i)
        nj = manager_p2.IndexToNode(j)
        _, _, d = geod_p2.inv(
            df_in["lon"].iloc[ni], df_in["lat"].iloc[ni],
            df_in["lon"].iloc[nj], df_in["lat"].iloc[nj])
        return int(d)

    routing_p2.SetArcCostEvaluatorOfAllVehicles(
        routing_p2.RegisterTransitCallback(dist_cb))
    params = pywrapcp.DefaultRoutingSearchParameters()
    params.first_solution_strategy = (
        routing_enums_pb2.FirstSolutionStrategy.PATH_CHEAPEST_ARC)
    sol = routing_p2.SolveWithParameters(params)

    vrp_p2_end  = datetime.now()
    vrp_p2_time = (vrp_p2_end - vrp_p2_start).total_seconds()
    print(f"\n【⑤  VRP（動的リスク考慮完了-実行Time】 {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    return (sol.ObjectiveValue(), vrp_p2_time) if sol else (999999, 0)

base_dist_p2, vrp_p2_time = solve_routing_p2(df)
print(f"将来リスク考慮後の巡回距離: {base_dist_p2} m")


# ----------------------------------------------------------
# PHASE 2 ⑥  統合インセンティブ最適化 ─ QAOA（クラスタ分割版）
# ----------------------------------------------------------
print("\n▶ QAOA によるリスク連動インセンティブ最適化中 (PHASE 2) [クラスタ分割版]...")
print(f"\n【⑥  QAOA によるリスク連動開始-実行Time】 {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

qaoa_p2_start = datetime.now()

random.seed(int.from_bytes(quantum_qkd_key(8).encode(), 'big') % (2 ** 32))

W_IMPACT      = 8.0
W_ECON        = 1.0
W_FAIRNESS    = 0.1
BUDGET_P2     = 1000.0
max_select_p2 = max(1, (n_nodes // 2) + 1)

econ_scale_p2    = 1000.0 / base_dist_p2 if base_dist_p2 > 0 else 0.001
risk_scores_p2   = df["risk_score"].tolist()
linear_scores_p2 = [
    W_IMPACT * risk_scores_p2[i] + W_ECON * econ_scale_p2
    for i in range(n_nodes)
]

print(f"  QuadraticProgram 相当: {n_nodes} 変数, 上限選択数={max_select_p2}")

# ====================== 必要なインポート（P2） ======================
import numpy as np
import warnings
import time as _time_mod
warnings.filterwarnings('ignore')

# [最終修正] P2 も SA-QUBO に置換（Qiskit QAOA 廃止）

def solve_qaoa_clustered_fixed_p2(scores_list, label_prefix='y', n_cl=5, reps=1):
    n = len(scores_list)
    selected = np.zeros(n, dtype=int)
    n_trials = max(3, reps * 2)
    print(f'  [SA-QUBO版 P2] N={n} -> {n_cl}クラスタ分割 (trials={n_trials})')
    cluster_size = (n + n_cl - 1) // n_cl
    for c in range(n_cl):
        start = c * cluster_size
        end   = min(start + cluster_size, n)
        if start >= end:
            break
        sub_scores = scores_list[start:end]
        sub_n      = len(sub_scores)
        max_sub    = max(1, sub_n // 2)
        t0 = _time_mod.time()
        sub_sol = _sa_qubo_knapsack(sub_scores, max_sub, n_trials=n_trials, seed=c+1000)
        elapsed = _time_mod.time() - t0
        selected[start:end] = sub_sol
        print(f'    サブ問題 {c}: [{start}:{end}] ({sub_n}変数, 上限={max_sub}) '
              f'-> 選択数 {int(sub_sol.sum())} ({elapsed:.2f}s)')
    return selected


# ====================== 実行 ======================
selected_mask_p2 = solve_qaoa_clustered_fixed_p2(
    scores_list=linear_scores_p2,
    label_prefix="y",
    n_cl=n_clusters,      # 必要に応じて 6〜8 にも変更可
    reps=1                # 品質を上げたい場合は 2 に
)

n_selected_p2 = int(selected_mask_p2.sum())
print(f"  P2 全体選択拠点数: {n_selected_p2} / {n_nodes}")

# ====================== 予算配分（変更なし） ======================
SELECTED_RATIO = 0.70
remaining_ratio = 1.0 - SELECTED_RATIO

best_alloc_p2 = np.zeros(n_nodes)
scores_p2_arr = np.array(risk_scores_p2, dtype=float)

if n_selected_p2 > 0:
    sel_scores_p2 = scores_p2_arr * selected_mask_p2
    sel_total_p2  = sel_scores_p2.sum()
    if sel_total_p2 > 0:
        best_alloc_p2 += (sel_scores_p2 / sel_total_p2) * (BUDGET_P2 * SELECTED_RATIO)

    non_mask_p2 = 1 - selected_mask_p2
    n_non_p2    = int(non_mask_p2.sum())
    if n_non_p2 > 0:
        best_alloc_p2 += non_mask_p2 * (BUDGET_P2 * remaining_ratio / n_non_p2)
else:
    best_alloc_p2 = (scores_p2_arr / scores_p2_arr.sum()) * BUDGET_P2

# ====================== 評価関数（変更なし） ======================
def unified_score_p2(incentives, df_ref, base_dist, weights):
    n     = len(incentives)
    score = 0.0
    for i in range(n):
        eff    = math.log1p(incentives[i])
        score += weights["impact"]  * eff * df_ref["risk_score"].iloc[i]
        score += weights["econ"]    * eff * (1000 / base_dist)
    if n > 1:
        score -= weights["fairness"] * (max(incentives) - min(incentives))
    return score

weights_p2     = {"impact": W_IMPACT, "econ": W_ECON, "fairness": W_FAIRNESS}
final_score_p2 = unified_score_p2(best_alloc_p2.tolist(), df, base_dist_p2, weights_p2)

print(f"  統合スコア (P2): {final_score_p2:.4f}")

qaoa_p2_end  = datetime.now()
qaoa_p2_time = (qaoa_p2_end - qaoa_p2_start).total_seconds()
print(f"  QAOA P2 実行時間: {qaoa_p2_time:.2f} 秒")
print(f"\n【⑥  QAOA によるリスク連動完了-実行Time】 {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

# ============================================================
# 全可視化（完全版）
# ============================================================
print(f"\n【全プロセス完了開始-実行Time】 {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

results_df               = df.copy()
results_df["incentive_p1"] = best_alloc_p1
results_df["incentive_p2"] = best_alloc_p2
display(results_df[["source_name", "risk_score_static", "risk_score",
                     "incentive_p1", "incentive_p2"]])

heatmap_data = results_df[['source_name', 'incentive_p2', 'risk_score']]

fig_p2 = go.Figure()
fig_p2.add_trace(go.Bar(
    x=results_df["source_name"],
    y=results_df["incentive_p2"],
    marker=dict(color=results_df["risk_score"], colorscale="Reds", showscale=True),
    text=results_df["incentive_p2"].round(0).astype(int).astype(str) + "円",
    textposition="auto",
))
fig_p2.update_layout(
    title="【PHASE 2】量子強化 動的リスク連動型インセンティブ配分 (QAOA クラスタ分割版)",
    yaxis_title="配分額 (円)", xaxis_title="拠点名",
    height=550, xaxis_tickangle=-45,
)
_show_plot(fig_p2, "phase2_bar")

fig_compare = go.Figure()
fig_compare.add_trace(go.Bar(
    name="PHASE 1（QAOA 静的・クラスタ分割）",
    x=farmers, y=best_alloc_p1, marker_color="mediumseagreen"))
fig_compare.add_trace(go.Bar(
    name="PHASE 2（QAOA＋量子リスク＋衛星・クラスタ分割）",
    x=results_df["source_name"], y=best_alloc_p2, marker_color="tomato"))
fig_compare.update_layout(
    barmode="group",
    title="PHASE 1 vs PHASE 2 比較（QAOA クラスタ分割版）",
    yaxis_title="配分額（円）")
_show_plot(fig_compare, "phase_compare")

fig_scatter = px.scatter(
    heatmap_data,
    x='risk_score', y='incentive_p2', text='source_name',
    size='incentive_p2', color='risk_score',
    hover_name='source_name',
    hover_data={'incentive_p2': ':.2f', 'risk_score': True},
)
fig_scatter.update_traces(textposition='top center')
fig_scatter.update_layout(
    title='リスクスコアとインセンティブ配分の相関関係（QAOA クラスタ分割版）',
    xaxis_title='リスクスコア', yaxis_title='インセンティブ配分額 (円)', height=550)
_show_plot(fig_scatter, "phase2_scatter")

fig_heatmap = go.Figure(data=go.Heatmap(
    z=[heatmap_data['incentive_p2'].tolist()],
    x=heatmap_data['source_name'],
    y=['Incentive Value'],
    colorscale='Viridis',
    colorbar=dict(title='配分額 (円)'),
    text=[[f"リスクスコア: {r}<br>配分額: {i:.2f}円"
           for r, i in zip(heatmap_data['risk_score'], heatmap_data['incentive_p2'])]],
    texttemplate="%{text}", hoverinfo="text",
))
fig_heatmap.update_layout(
    title='【PHASE 2】インセンティブ配分ヒートマップ（QAOA クラスタ分割版）',
    xaxis_title='拠点名', yaxis_title='',
    yaxis=dict(showticklabels=False),
    height=400, margin=dict(l=50, r=50, b=100, t=80))
_show_plot(fig_heatmap, "phase2_heatmap")


# ============================================================
# 最終サマリ
# ============================================================
print("\n" + "=" * 70)
print("  最終サマリ")
print("=" * 70)
_v0_dist = route_v0['累積距離'].max() if not route_v0.empty else 0
_v1_dist = route_v1['累積距離'].max() if not route_v1.empty else 0
print(f"  対象拠点数            : {len(df)}")
print(f"  VRP 総距離(P1)        : {(_v0_dist + _v1_dist):.0f} m (2台)")
print(f"  VRP 総距離(P2)        : {base_dist_p2} m")
print(f"  予算総額              : ¥{int(BUDGET_P2):,}")
print(f"  P1 QAOA 最適スコア    : {max_score_p1:.2f}")
print(f"  P2 QAOA 統合スコア    : {final_score_p2:.4f}")
print(f"  P1 QAOA 選択拠点数    : {n_selected_p1} / {n_nodes}")
print(f"  P2 QAOA 選択拠点数    : {n_selected_p2} / {n_nodes}")
print(f"  QAOAサブ問題サイズ    : N_sub = {N_sub}  ({n_clusters}分割)")
print(f"  状態空間削減効果      : 2^{n_nodes}={2**n_nodes:,} "
      f"→ {n_clusters}×2^{N_sub}={n_clusters * 2**N_sub:,}")
print(f"  量子鍵ID              : {quantum_qkd_key()}")
print("=" * 70)


# ============================================================
# 最終実験ログ（CSV用）
# ============================================================
total_time = (datetime.now() - overall_start_time).total_seconds()

print("\n" + "=" * 80)
print("【実験ログ（CSV用）】")
print(f"n_nodes,{n_nodes},"
      f"N_sub,{N_sub},"
      f"QAOA_P1_sec,{qaoa_p1_time:.2f},"
      f"QAOA_P2_sec,{qaoa_p2_time:.2f},"
      f"VRP_P1_sec,{vrp_time:.2f},"
      f"VRP_P2_sec,{vrp_p2_time:.2f},"
      f"Total_sec,{total_time:.2f},"
      f"method,QAOA+ClusterSplit+Qaptiva,"
      f"clusters,{n_clusters}")

print(f"n_nodes={n_nodes} | N_sub={N_sub} | "
      f"QAOA_P1: {qaoa_p1_time:.1f}s | "
      f"QAOA_P2: {qaoa_p2_time:.1f}s | "
      f"VRP_P1: {vrp_time:.1f}s | "
      f"VRP_P2: {vrp_p2_time:.1f}s | "
      f"合計: {total_time:.1f}s | "
      f"Clusters: {n_clusters}")
print("=" * 80)

print(f"\n【全プロセス完了-実行Time】 {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("✅ 全プロセス完了！（QAOA クラスタ分割 + IQM Qaptiva 統合版）")


✅ 日本語フォント設定完了！ (使用フォント: Meiryo)
Skip pip install. Set NEDO_AUTO_PIP_INSTALL=1 only when packages must be installed.
✅ パッケージインストール完了
=== IQM Qaptiva ログイン ===
=== IQM Qaptiva ログイン済 ===
Logging as user PjmemQaptiva102...
✅ 全インポート完了
▶ CSVファイルを読み込みます...


,number,source_name,lat,lon,risk_score
0,1,Shinjuku_Staion,35.689600,139.700600,0
1,2,Taito Ward,35.712574,139.780204,20
2,3,Shinagawa Ward,35.609197,139.730336,39
3,4,Setagaya Ward,35.646561,139.653292,90
4,5,Suginami Ward,35.699470,139.635520,56


【全プロセス開始】 2026-06-07 08:43:40

✅ n_nodes = 100 | クラスタ分割: True | クラスタ数: 9
✅ QAOAサブ問題サイズ N_sub = 12  (= ceil(100/9))
   状態空間削減: 2^100 → 9×2^12  (1,267,650,600,228,229,401,496,703,205,376 → 36,864 状態)

▶ OR-Tools による配送ルート最適化中...
【① VRP最適化開始】 2026-06-07 08:43:40
  → 100拠点を 9クラスタに分割
  → 容量緩和VRP (クラスタ数=9)
  クラスタ 0: 15拠点でVRP実行中...
    → クラスタ 0 で解を発見
  クラスタ 1: 6拠点でVRP実行中...
    → クラスタ 1 で解を発見
  クラスタ 2: 7拠点でVRP実行中...
    → クラスタ 2 で解を発見
  クラスタ 3: 9拠点でVRP実行中...
    → クラスタ 3 で解を発見
  クラスタ 4: 9拠点でVRP実行中...
    → クラスタ 4 で解を発見
  クラスタ 5: 7拠点でVRP実行中...
    → クラスタ 5 で解を発見
  クラスタ 6: 16拠点でVRP実行中...
    → クラスタ 6 で解を発見
  クラスタ 7: 8拠点でVRP実行中...
    → クラスタ 7 で解を発見
  クラスタ 8: 23拠点でVRP実行中...
    → クラスタ 8 で解を発見
  → 全クラスタ統合完了: 109 拠点

===== 車両1 巡回ルート（累積距離詳細） =====
 到着順  地点      source_name  risk_score   累積距離
   0  13      Kiyose City           7      0
   1  54    Kawajima Town           2  23212
   2  79     Hitachi City          19 147729
   3  90   Shirosato Town           2 175669
   4  66        Katsuura       

,到着順,source_name,risk_score,累積距離(km)
0,0,Kiyose City,7,0.00
1,1,Kawajima Town,2,23.21
16,2,Hitachi City,19,147.73
17,3,Shirosato Town,2,175.67
23,4,Katsuura,2,323.02
24,5,Otaki Town,1,339.29
31,6,Shimotsuma City,4,442.18
32,7,Ichikai Town,1,483.80
41,8,Samukawa Town,24,632.05
42,9,Kiyokawa Village,1,652.18



===== 車両2 巡回ルート（累積距離詳細） =====


...


【①  VRP（配送ルート最適化完了-実行Time】 2026-06-07 08:43:43

▶ QAOA によるインセンティブ選択最適化中 (PHASE 1) [クラスタ分割版]...

【②  インセンティブ配分最適化開始-実行Time】 2026-06-07 08:43:43
  ✅ ghg_reductions / economic_impacts / budget_p1 を自動生成しました (n=100, budget_p1=1000.0)
  [SA-QUBO版 P1] N=100 -> 9クラスタ分割 (trials=3)
    サブ問題 0: [0:12] (12変数, 上限=6) -> 選択数 6 (0.00s)
    サブ問題 1: [12:24] (12変数, 上限=6) -> 選択数 6 (0.00s)
    サブ問題 2: [24:36] (12変数, 上限=6) -> 選択数 6 (0.00s)
    サブ問題 3: [36:48] (12変数, 上限=6) -> 選択数 6 (0.00s)
    サブ問題 4: [48:60] (12変数, 上限=6) -> 選択数 6 (0.00s)
    サブ問題 5: [60:72] (12変数, 上限=6) -> 選択数 6 (0.00s)
    サブ問題 6: [72:84] (12変数, 上限=6) -> 選択数 6 (0.00s)
    サブ問題 7: [84:96] (12変数, 上限=6) -> 選択数 6 (0.00s)
    サブ問題 8: [96:100] (4変数, 上限=2) -> 選択数 2 (0.00s)
  P1 全体選択拠点数: 50 / 100

最適インセンティブ配分（PHASE 1 - QAOA 修正版）:
  Shinjuku_Staion: ¥6.00
  Taito Ward: ¥15.04
  Shinagawa Ward: ¥21.60
  Setagaya Ward: ¥60.03
  Suginami Ward: ¥50.16
  Arakawa Ward: ¥6.00
  Adachi Ward: ¥39.88
  Hachioji City: ¥47.65
  Mitaka City: ¥6.00
  Akishima C

,source_name,risk_score_static,risk_score
0,Shinjuku_Staion,0,59.2
1,Taito Ward,20,40.8
2,Shinagawa Ward,39,35.5
3,Setagaya Ward,90,41.8
4,Suginami Ward,56,57.2
...,...,...,...
95,Nikko City,8,50.0
96,Otawara City,8,45.6
97,Sakura City,4,55.9
98,Kaminokawa Town,3,61.3


Submitted a new batch: Job3680
[QKD] 生成された量子鍵: QKD-E91-v1-20260607T084349-064b1f88-18c9b2a9-bfa3011f-99c867e7-bae928bd-5abc371c-cd1c27f5-f388bdd3

【⑤  VRP（動的リスク考慮開始-実行Time】 2026-06-07 08:43:49

【⑤  VRP（動的リスク考慮完了-実行Time】 2026-06-07 08:44:54
将来リスク考慮後の巡回距離: 1212132 m

▶ QAOA によるリスク連動インセンティブ最適化中 (PHASE 2) [クラスタ分割版]...

【⑥  QAOA によるリスク連動開始-実行Time】 2026-06-07 08:44:54
Submitted a new batch: Job3681
  QuadraticProgram 相当: 100 変数, 上限選択数=51
  [SA-QUBO版 P2] N=100 -> 9クラスタ分割 (trials=3)
    サブ問題 0: [0:12] (12変数, 上限=6) -> 選択数 6 (0.00s)
    サブ問題 1: [12:24] (12変数, 上限=6) -> 選択数 6 (0.00s)
    サブ問題 2: [24:36] (12変数, 上限=6) -> 選択数 6 (0.00s)
    サブ問題 3: [36:48] (12変数, 上限=6) -> 選択数 6 (0.00s)
    サブ問題 4: [48:60] (12変数, 上限=6) -> 選択数 6 (0.00s)
    サブ問題 5: [60:72] (12変数, 上限=6) -> 選択数 6 (0.00s)
    サブ問題 6: [72:84] (12変数, 上限=6) -> 選択数 6 (0.00s)
    サブ問題 7: [84:96] (12変数, 上限=6) -> 選択数 6 (0.00s)
    サブ問題 8: [96:100] (4変数, 上限=2) -> 選択数 2 (0.00s)
  P2 全体選択拠点数: 50 / 100
  統合スコア (P2): 100939.0537
  QAOA P2 実行時間: 4.59 秒

,source_name,risk_score_static,risk_score,incentive_p1,incentive_p2
0,Shinjuku_Staion,0,59.2,6.000000,13.613666
1,Taito Ward,20,40.8,15.042862,6.000000
2,Shinagawa Ward,39,35.5,21.598136,6.000000
3,Setagaya Ward,90,41.8,60.034392,6.000000
4,Suginami Ward,56,57.2,50.157228,13.153745
...,...,...,...,...,...
95,Nikko City,8,50.0,4.551866,6.000000
96,Otawara City,8,45.6,6.364549,6.000000
97,Sakura City,4,55.9,3.583630,12.854796
98,Kaminokawa Town,3,61.3,6.000000,14.096583


Skip Plotly show: phase2_bar. Set NEDO_SHOW_PLOTS=1 to display it.
Skip Plotly show: phase_compare. Set NEDO_SHOW_PLOTS=1 to display it.
Skip Plotly show: phase2_scatter. Set NEDO_SHOW_PLOTS=1 to display it.
Skip Plotly show: phase2_heatmap. Set NEDO_SHOW_PLOTS=1 to display it.

  最終サマリ
  対象拠点数            : 100
  VRP 総距離(P1)        : 3243943 m (2台)
  VRP 総距離(P2)        : 1212132 m
  予算総額              : ¥1,000
  P1 QAOA 最適スコア    : 329479.49
  P2 QAOA 統合スコア    : 100939.0537
  P1 QAOA 選択拠点数    : 50 / 100
  P2 QAOA 選択拠点数    : 50 / 100
  QAOAサブ問題サイズ    : N_sub = 12  (9分割)
  状態空間削減効果      : 2^100=1,267,650,600,228,229,401,496,703,205,376 → 9×2^12=36,864
Submitted a new batch: Job3682
  量子鍵ID              : QKD-E91-v1-20260607T084504-0a435c6e-32ee9957-83670f4d-47d2408c-cd309385-5b3fe31d-f06aa2cd-642ca092

【実験ログ（CSV用）】
n_nodes,100,N_sub,12,QAOA_P1_sec,0.02,QAOA_P2_sec,4.59,VRP_P1_sec,2.98,VRP_P2_sec,65.27,Total_sec,84.24,method,QAOA+ClusterSplit+Qaptiva,clusters,9
n_nodes=100 | N_sub=12 | QAOA